# The Geometry of the Covariance Matrix

This notebook explores the geometric interpretation of the Covariance Matrix in 2D space. By treating statistics through the lens of linear algebra, we can visualize how data distributions are formed through linear transformations.

---

## 1. The Mathematical Definition

Given a dataset $X$ that has been **centered** (mean shifted to 0), the Covariance Matrix $\Sigma$ is defined as:

$$
\Sigma = \frac{1}{N-1} X^T X = \begin{bmatrix} \sigma^2_x & \sigma_{xy} \\ \sigma_{yx} & \sigma^2_y \end{bmatrix}
$$

### Key Components:
* **$\sigma^2_x$ (Top-Left):** Variance of feature $X$. Represents the spread along the x-axis.
* **$\sigma^2_y$ (Bottom-Right):** Variance of feature $Y$. Represents the spread along the y-axis.
* **$\sigma_{xy}$ (Off-Diagonal):** Covariance between $X$ and $Y$. Note that $\sigma_{xy} = \sigma_{yx}$ (The matrix is symmetric).
* **$\frac{1}{N-1}$:** **Bessel's Correction**. This adjusts the calculation to provide an unbiased estimate of the population variance by accounting for the degree of freedom lost when calculating the sample mean.

---

## 2. Geometric Intuition: The Linear Transformation

To visualize $\Sigma$, imagine starting with a **Standard Normal Distribution** (a unit circle, often called "White Noise").
* **Standard Basis:** $\hat{i} = (1,0)$, $\hat{j} = (0,1)$
* **Variance:** 1 in all directions.
* **Correlation:** 0.

The Covariance Matrix describes how to **stretch** and **rotate** this unit circle to match the shape of your data.

> **Note:** To generate points that match a specific covariance $\Sigma$, we multiply the standard normal data by the matrix square root (Cholesky decomposition) $L$, where $L = \sqrt{\Sigma}$.

---

## 3. Visualizing the Parameters

| Matrix Element | Visual Effect |
| :--- | :--- |
| **Diagonal Values** ($\sigma^2$) | **Scaling.** High values stretch the cloud along the axes. Low values compress it. |
| **Off-Diagonal Values** ($\sigma_{xy}$) | **Rotation.** Non-zero values tilt the cloud off the main axes. |
| **Positive Covariance** | Tilts the cloud towards the $y=x$ diagonal ($/$). |
| **Negative Covariance** | Tilts the cloud towards the $y=-x$ diagonal ($\setminus$). |

---

## 4. Scenario Analysis

Below is a breakdown of 10 distinct geometric scenarios dependent on the values within $\Sigma$.

| Case | Matrix $\Sigma$ | Geometry Description |
| :--- | :--- | :--- |
| **1. Standard Normal** | $\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$ | **Perfect Circle.** No correlation. Equal spread in all directions. Basis vectors remain orthogonal. |
| **2. High Variance** | $\begin{bmatrix} 3 & 0 \\ 0 & 3 \end{bmatrix}$ | **Large Circle.** Uniform expansion. The area of uncertainty increases, but the shape remains circular. |
| **3. X-Dominant** | $\begin{bmatrix} 4 & 0 \\ 0 & 0.5 \end{bmatrix}$ | **Flat Ellipse.** Severe horizontal stretching (High $\sigma_x^2$) and vertical compression. |
| **4. Y-Dominant** | $\begin{bmatrix} 0.5 & 0 \\ 0 & 4 \end{bmatrix}$ | **Tall Ellipse.** Severe vertical stretching (High $\sigma_y^2$). Major axis aligns with Y. |
| **5. Weak Pos. Corr** | $\begin{bmatrix} 2 & 0.8 \\ 0.8 & 2 \end{bmatrix}$ | **Slight Tilt (/).** The cloud begins to rotate towards the line $y=x$. Basis vectors lose orthogonality. |
| **6. Strong Pos. Corr** | $\begin{bmatrix} 4 & 3.5 \\ 3.5 & 4 \end{bmatrix}$ | **Tight Diagonal (/).** High linearity implies $x \approx y$. The ellipse is very narrow. |
| **7. Weak Neg. Corr** | $\begin{bmatrix} 2 & -0.8 \\ -0.8 & 2 \end{bmatrix}$ | **Slight Tilt (\).** The cloud rotates towards $y=-x$. Inverse relationship emerging. |
| **8. Strong Neg. Corr** | $\begin{bmatrix} 4 & -3.5 \\ -3.5 & 4 \end{bmatrix}$ | **Tight Diagonal (\).** Strong inverse dependency. X increases as Y decreases substantially. |
| **9. Anisotropic** | $\begin{bmatrix} 5 & 2 \\ 2 & 1 \end{bmatrix}$ | **Sheared / Skewed.** Uneven stretching combined with rotation. High X variance, low Y variance, positive correlation. |
| **10. Near Singular** | $\begin{bmatrix} 2 & 1.95 \\ 1.95 & 2 \end{bmatrix}$ | **1D Line.** The determinant $|\Sigma| \approx 0$. The ellipse collapses, indicating redundancy in the data dimensions. |

---

## 5. Eigenvectors & Principal Components

In the context of the geometry:

* **Eigenvectors:** Represent the **directions** of the ellipse's principal axes (the widest and narrowest parts). They define the natural coordinate system of the data.
* **Eigenvalues:** Represent the **variance** (magnitude squared) along those principal directions.

$$\Sigma v = \lambda v$$

Where $v$ is the eigenvector (direction of spread) and $\lambda$ is the eigenvalue (amount of spread).

In [10]:
from manim import *
import numpy as np
from scipy.linalg import sqrtm

class CovarianeExplained(Scene):
    def construct(self):
        # --- CONFIGURATION ---
        self.camera.background_color = "#1a1a1a"
        
        # --- PART 1: INTRODUCTION & FORMULAE ---
        self.intro_and_formulae()
        
        # --- PART 2: BASIS TRANSFORMATION EXPLANATION ---
        self.explain_basis_transformation()
        
        # --- PART 3: 10 DETAILED EXAMPLES ---
        self.ten_covariance_examples()

    def intro_and_formulae(self):
        # Title - Top Left Corner
        title = Text("Covariance Matrix in 2D", font_size=42, color=BLUE)
        title.to_edge(UP)
        self.play(Write(title))
        self.wait(1)
        # Move title to corner to make space
        self.play(title.animate.scale(0.7).to_corner(UL))

        # Definition Text
        def_text = Tex(r"Given centered data $X$, the Covariance Matrix $\Sigma$ is:", font_size=32)
        def_text.next_to(title, DOWN, aligned_edge=LEFT, buff=0.5)
        
        # Formula 
        formula = MathTex(
            r"\Sigma = \frac{1}{N-1} X^T X = \begin{bmatrix} \sigma^2_x & \sigma_{xy} \\ \sigma_{yx} & \sigma^2_y \end{bmatrix}",
            font_size=38
        )
        formula.next_to(def_text, DOWN, buff=0.4, aligned_edge=LEFT)

        self.play(Write(def_text))
        self.play(Write(formula))
        self.wait(2)

        # Geometric Intuition Text
        geo_text = VGroup(
            Text("Geometric Interpretation:", font_size=28, color=GREEN),
            Tex(r"$\bullet$ Diagonals ($\sigma^2$): Axis stretching (Variance)", font_size=28),
            Tex(r"$\bullet$ Off-Diagonals ($\sigma_{xy}$): Rotation (Correlation)", font_size=28),
            Tex(r"Transformation $L = \sqrt{\Sigma}$ maps:", font_size=28),
            Tex(r"Unit Circle $\rightarrow$ Data Ellipse", font_size=28, color=YELLOW)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.15)
        geo_text.next_to(formula, DOWN, buff=0.5, aligned_edge=LEFT)

        self.play(Write(geo_text))
        self.wait(2)
        
        # Group for cleanup
        self.intro_group = VGroup(def_text, formula, geo_text)

    def explain_basis_transformation(self):
        # Clear left side text
        self.play(FadeOut(self.intro_group))
        
        # Explanation of Basis
        explanation = VGroup(
            Text("The Basis Vectors", font_size=32, color=BLUE),
            Tex(r"Standard Basis: $\hat{i}=(1,0), \hat{j}=(0,1)$", font_size=28),
            Text("Covariance determines how", font_size=28),
            Text("these vectors stretch & rotate.", font_size=28),
            Tex(r"Length $\propto$ Variance", font_size=28, color=YELLOW),
            Tex(r"Angle $\propto$ Correlation", font_size=28, color=ORANGE)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.2)
        explanation.to_edge(LEFT).shift(RIGHT*0.2)
        
        self.play(Write(explanation))

        # Setup Grid on Right Side
        grid = NumberPlane(
            x_range=[-4, 4, 1],
            y_range=[-4, 4, 1],
            background_line_style={"stroke_color": TEAL, "stroke_opacity": 0.4}
        ).scale(0.8).to_edge(RIGHT)

        # Basis Vectors
        i_hat = Arrow(start=grid.get_origin(), end=grid.c2p(1,0), color=RED, buff=0)
        j_hat = Arrow(start=grid.get_origin(), end=grid.c2p(0,1), color=YELLOW, buff=0)
        
        i_label = MathTex(r"\hat{i}", color=RED).next_to(i_hat, DOWN, buff=0.1)
        j_label = MathTex(r"\hat{j}", color=YELLOW).next_to(j_hat, LEFT, buff=0.1)

        self.play(Create(grid), GrowArrow(i_hat), GrowArrow(j_hat), Write(i_label), Write(j_label))
        self.wait(1)

        # Example Transformation Matrix
        cov_ex = np.array([[2.0, 1.2], [1.2, 2.0]])
        transform_matrix = np.real(sqrtm(cov_ex))

        # Display Matrix
        matrix_tex = MathTex(
            r"\Sigma_{demo} = \begin{bmatrix} 2.0 & 1.2 \\ 1.2 & 2.0 \end{bmatrix}", 
            color=BLUE, font_size=36
        ).next_to(explanation, DOWN, buff=0.5)
        self.play(Write(matrix_tex))

        # Animate Transformation
        self.play(
            grid.animate.apply_matrix(transform_matrix),
            i_hat.animate.put_start_and_end_on(grid.get_origin(), grid.c2p(*transform_matrix[:,0])),
            j_hat.animate.put_start_and_end_on(grid.get_origin(), grid.c2p(*transform_matrix[:,1])),
            i_label.animate.move_to(grid.c2p(*transform_matrix[:,0]) + DOWN*0.3),
            j_label.animate.move_to(grid.c2p(*transform_matrix[:,1]) + LEFT*0.3),
            run_time=2.5
        )
        self.wait(1)

        # Clean up
        self.play(
            FadeOut(explanation), 
            FadeOut(matrix_tex), 
            FadeOut(grid), 
            FadeOut(i_hat), FadeOut(j_hat), 
            FadeOut(i_label), FadeOut(j_label)
        )

    def ten_covariance_examples(self):
        # 10 Scenarios with DETAILED GEOMETRIC EXPLANATIONS
        # Format: (Title, Matrix, List of Geometry Description Lines)
        cases = [
            ("1. Standard Normal", [[1, 0], [0, 1]], 
             ["Perfectly circular distribution.", "Equal variance in X and Y axes.", "Basis vectors preserve orthogonality."]),
            
            ("2. High Variance", [[3, 0], [0, 3]], 
             ["Uniform expansion in all directions.", "Area increases by determinant.", "No rotation (Uncorrelated)."]),
            
            ("3. X-Dominant", [[4, 0], [0, 0.5]], 
             ["Severe stretching along X-axis.", "Compression along Y-axis.", "Ellipse flattens horizontally."]),
            
            ("4. Y-Dominant", [[0.5, 0], [0, 4]], 
             ["Stretched vertically (High Y var).", "X-axis variance is minimal.", "Major axis aligns with Y."]),
            
            ("5. Weak Pos. Correlation", [[2, 0.8], [0.8, 2]], 
             ["Slight tilt towards $y=x$ line.", "Positive covariance pulls axes.", "Basis vectors lose orthogonality."]),
            
            ("6. Strong Pos. Correlation", [[4, 3.5], [3.5, 4]], 
             ["Tight ellipse along diagonal.", "High linearity implies $x \\approx y$.", "Eigenvectors rotated 45 degrees."]),
            
            ("7. Weak Neg. Correlation", [[2, -0.8], [-0.8, 2]], 
             ["Slight tilt towards $y=-x$ line.", "Inverse relationship emerging.", "X increases as Y decreases."]),
            
            ("8. Strong Neg. Correlation", [[4, -3.5], [-3.5, 4]], 
             ["Narrow shape sloping downwards.", "Strong inverse dependency.", "Major axis along 135 degrees."]),
            
            ("9. Anisotropic Positive", [[5, 2], [2, 1]], 
             ["Uneven stretching (High X, Low Y).", "Rotation due to covariance term.", "Complex shear transformation."]),
            
            ("10. Near Singular", [[2, 1.95], [1.95, 2]], 
             ["Collapse into a 1D line.", "Determinant approaches zero.", "Loss of dimension/information."])
        ]

        # Tracking variables for smooth transitions
        current_info_group = None
        current_grid = None
        current_dots = None
        current_vectors = None

        for i, (label, cov_matrix, geo_desc_lines) in enumerate(cases):
            # 1. Prepare Left Side Text (Top Space)
            title_text = Text(label, font_size=28, color=YELLOW)
            
            # Matrix formatting
            mat_str = r"\begin{bmatrix} " + f"{cov_matrix[0][0]:.1f}" + r" & " + f"{cov_matrix[0][1]:.1f}" + r" \\ " + \
                      f"{cov_matrix[1][0]:.1f}" + r" & " + f"{cov_matrix[1][1]:.1f}" + r" \end{bmatrix}"
            math_mat = MathTex(r"\Sigma = " + mat_str, font_size=40)
            
            # Detailed Geometry Description (Multi-line)
            desc_group = VGroup()
            # Font size 32 for Header
            desc_label = Text("Geometry:", font_size=32, color=WHITE, weight=BOLD)
            desc_group.add(desc_label)
            
            for line in geo_desc_lines:
                # Font size 32 for Lines
                l = Tex(r"$\cdot$ " + line, font_size=32, color=BLUE_B)
                desc_group.add(l)
            
            desc_group.arrange(DOWN, aligned_edge=LEFT, buff=0.15)

            # Grouping - Top Left
            info_group = VGroup(title_text, math_mat, desc_group).arrange(DOWN, aligned_edge=LEFT, buff=0.35)
            # MOVED DOWN by 0.7 (0.3 original + 0.4 requested)
            info_group.to_corner(UL).shift(DOWN*0.7) 

            # 2. Prepare Right Side Visuals
            L = np.real(sqrtm(np.array(cov_matrix)))
            
            grid = NumberPlane(
                x_range=[-6, 6], y_range=[-6, 6],
                background_line_style={"stroke_opacity": 0.3}
            ).scale(0.65).to_edge(RIGHT)
            
            # Data Cloud Generation
            np.random.seed(42 + i)
            standard_points = np.random.randn(300, 2) 
            transformed_points = standard_points @ L.T
            
            dots = VGroup()
            for p in transformed_points:
                dots.add(Dot(grid.c2p(p[0], p[1]), radius=0.03, color=TEAL))

            # Eigenvectors
            eig_vals, eig_vecs = np.linalg.eigh(np.array(cov_matrix))
            
            # Visual Vectors
            v1_arr = eig_vecs[:, 0] * np.sqrt(abs(eig_vals[0])) * 2
            v2_arr = eig_vecs[:, 1] * np.sqrt(abs(eig_vals[1])) * 2
            
            vec1 = Arrow(grid.get_origin(), grid.c2p(v1_arr[0], v1_arr[1]), color=ORANGE, buff=0)
            vec2 = Arrow(grid.get_origin(), grid.c2p(v2_arr[0], v2_arr[1]), color=PURPLE, buff=0)
            vectors = VGroup(vec1, vec2)

            # 3. Animation Sequence
            if current_info_group:
                self.play(
                    FadeOut(current_info_group), 
                    FadeOut(current_grid), 
                    FadeOut(current_dots), 
                    FadeOut(current_vectors),
                    run_time=0.5
                )
            
            self.play(FadeIn(info_group), run_time=0.5)
            self.play(Create(grid), run_time=0.5)
            self.play(FadeIn(dots), run_time=0.8)
            self.play(GrowArrow(vec1), GrowArrow(vec2), run_time=0.5)
            
            self.wait(2.0)
            
            current_info_group = info_group
            current_grid = grid
            current_dots = dots
            current_vectors = vectors

        # Final Cleanup
        self.play(
            FadeOut(current_info_group), 
            FadeOut(current_grid), 
            FadeOut(current_dots), 
            FadeOut(current_vectors)
        )

        final_text = Text("Covariance Matrix encodes\nthe Geometry of Uncertainty", font_size=36, color=BLUE)
        self.play(Write(final_text))
        self.wait(3)


%manim -qk -v warning CovarianeExplained

Manim Community v0.19.0